In [0]:
"""AI Gateway load generation notebook.

Calls three AI Gateway endpoints with randomly generated Canadian airline topics.
Uses the workspace-native OpenAI client for auth.
"""
import time
import random
from openai import OpenAI

# Databricks workspace-native OpenAI client
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://adb-984752964297111.11.azuredatabricks.net/ai-gateway/mlflow/v1",
)

# Endpoints to rotate through
ENDPOINTS = ["nova-shm", "gpt-nano-shm", "shm_endpoint"]

# QPM control
QPM = 30  # queries per minute target (adjust as needed)
DELAY = 60.0 / QPM

# Seed Canadian airline companies
CANADIAN_AIRLINES = [
    "Air Canada",
    "WestJet",
    "Porter Airlines",
]

print(f"Configured {len(ENDPOINTS)} endpoints at {QPM} QPM ({DELAY:.1f}s delay)")
print(f"Seed companies: {len(CANADIAN_AIRLINES)}")

In [0]:
def generate_topics(n_topics: int = 20, temperature: float = 1.5) -> list[str]:
    """Generate diverse Canadian airline topics using a high-temperature LLM call.
    
    Uses one of the endpoints to brainstorm topics, ensuring variety
    by seeding with random company names.
    """
    # Pick a random subset of companies to seed the prompt
    seed_companies = random.sample(CANADIAN_AIRLINES, min(5, len(CANADIAN_AIRLINES)))
    
    prompt = (
        f"Generate exactly {n_topics} diverse, specific discussion topics about Canadian airlines. "
        f"Make sure to include topics mentioning these companies: {', '.join(seed_companies)}. "
        "Topics should cover: routes, customer service, loyalty programs, fleet, pricing, "
        "regional connectivity, competition, weather delays, mergers, sustainability, "
        "in-flight experience, cargo operations, and labour relations. "
        "Return ONLY a numbered list, one topic per line."
    )
    
    raw = None
    try:
        response = client.chat.completions.create(
            model="databricks-claude-haiku-4-5",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=1000,
        )
        raw = response.choices[0].message.content
    except Exception as e:
        print(f"\u26a0\ufe0f [databricks-claude-haiku-4-5] Failed: {e}\n")
    
    if raw is None:
        print("\u26a0\ufe0f All endpoints blocked for topic generation. Using fallback topics.")
        templates = [
            "{company}'s route expansion strategy for 2026",
            "Customer loyalty program comparison: {company} vs competitors",
            "{company}'s fleet modernization and sustainability goals",
            "How {company} handles weather delays and cancellations",
            "Labour relations and pilot negotiations at {company}",
            "{company}'s cargo operations and growth potential",
            "In-flight experience and service quality at {company}",
            "Regional connectivity challenges for {company}",
            "{company}'s pricing strategy in the Canadian market",
            "The future of mergers and acquisitions involving {company}",
        ]
        topics = [
            t.format(company=c)
            for c in CANADIAN_AIRLINES
            for t in random.sample(templates, min(n_topics // len(CANADIAN_AIRLINES) + 1, len(templates)))
        ][:n_topics]
        print(f"Generated {len(topics)} fallback topics")
        for i, t in enumerate(topics[:5], 1):
            print(f"  {i}. {t}")
        if len(topics) > 5:
            print(f"  ... and {len(topics) - 5} more")
        return topics
    
    topics = [
        line.split(".", 1)[-1].strip().lstrip("-").strip()
        for line in raw.strip().split("\n")
        if line.strip() and any(c.isalpha() for c in line)
    ]
    
    if not topics:
        print("\u26a0\ufe0f LLM returned empty/unparseable response. Using fallback topics.")
        templates = [
            "{company}'s route expansion strategy for 2026",
            "Customer loyalty program comparison: {company} vs competitors",
            "{company}'s fleet modernization and sustainability goals",
            "How {company} handles weather delays and cancellations",
            "Labour relations and pilot negotiations at {company}",
            "{company}'s cargo operations and growth potential",
            "In-flight experience and service quality at {company}",
            "Regional connectivity challenges for {company}",
            "{company}'s pricing strategy in the Canadian market",
            "The future of mergers and acquisitions involving {company}",
        ]
        topics = [
            t.format(company=c)
            for c in CANADIAN_AIRLINES
            for t in random.sample(templates, min(n_topics // len(CANADIAN_AIRLINES) + 1, len(templates)))
        ][:n_topics]

    print(f"Generated {len(topics)} topics (temp={temperature})")
    for i, t in enumerate(topics[:5], 1):
        print(f"  {i}. {t}")
    if len(topics) > 5:
        print(f"  ... and {len(topics) - 5} more")
    
    return topics

topics = generate_topics(n_topics=30, temperature=1.0)

In [0]:
import concurrent.futures
from datetime import datetime

def call_endpoint(endpoint: str, topic: str) -> dict:
    """Make a single chat completion call and return metadata."""
    messages = [
        {"role": "system", "content": "You are a Canadian aviation industry analyst. Be concise."},
        {"role": "user", "content": topic},
    ]
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=endpoint,
            messages=messages,
            temperature=0.7,
            max_tokens=150,
        )
        elapsed = time.time() - start
        reply = response.choices[0].message.content
        return {
            "endpoint": endpoint,
            "request": topic,
            "response": reply,
            "status": "success",
            "latency_s": round(elapsed, 2),
            "tokens": response.usage.total_tokens if response.usage else None,
            "timestamp": datetime.now().isoformat(),
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            "endpoint": endpoint,
            "request": topic,
            "response": None,
            "status": f"error: {str(e)[:100]}",
            "latency_s": round(elapsed, 2),
            "tokens": None,
            "timestamp": datetime.now().isoformat(),
        }


def run_load_test(topics: list[str], qpm: int = QPM, rounds: int = 1) -> list[dict]:
    """Send topics to endpoints in round-robin, respecting QPM."""
    delay = 60.0 / qpm
    results = []
    
    for r in range(rounds):
        random.shuffle(topics)
        for i, topic in enumerate(topics):
            endpoint = ENDPOINTS[i % len(ENDPOINTS)]
            result = call_endpoint(endpoint, topic)
            results.append(result)
            
            status_icon = "✅" if result["status"] == "success" else "❌"
            print(f"{status_icon} [{result['endpoint']}] {result['latency_s']}s")
            print(f"   REQUEST:  {result['request'][:100]}")
            print(f"   RESPONSE: {(result['response'] or result['status'])[:150]}")
            print()
            
            time.sleep(delay)
    
    return results

# Run the load test
print(f"Starting load test: {len(topics)} topics x {len(ENDPOINTS)} endpoints @ {QPM} QPM")
print("=" * 80)
results = run_load_test(topics, qpm=QPM, rounds=2)

In [0]:
import pandas as pd

df_results = pd.DataFrame(results)

print("\n" + "=" * 80)
print("LOAD TEST SUMMARY")
print("=" * 80)
print(f"Total calls: {len(df_results)}")

if df_results.empty or "status" not in df_results.columns:
    print("No calls were made (topics list was empty or all calls failed to record).")
else:
    print(f"Successes: {(df_results['status'] == 'success').sum()}")
    print(f"Errors: {(df_results['status'] != 'success').sum()}")

if not df_results.empty and "status" in df_results.columns and (df_results["status"] == "success").any():
    summary = (
        df_results[df_results["status"] == "success"]
        .groupby("endpoint")
        .agg(
            calls=("latency_s", "count"),
            avg_latency=("latency_s", "mean"),
            p95_latency=("latency_s", lambda x: x.quantile(0.95)),
            avg_tokens=("tokens", "mean"),
        )
        .round(2)
    )
    display(summary)
else:
    print("No successful calls to summarize.")